In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage



C:\Users\sarka\AppData\Local\Temp\ipykernel_24256\3330463389.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
# load doc:
docs = PyPDFLoader('../data/SDE_Resume_Sandip.pdf').load()

# Split docs in chunks:
split_docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

# vector embbedings:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# store in vector DB:
vector_store = InMemoryVectorStore.from_documents(
    documents=split_docs,
    embedding=embeddings
    )

In [4]:
res = vector_store.similarity_search("what is the candidate name?")

In [5]:
### agent flow: tools, llms, prompt
@tool
def get_context(query: str):
    """ This tool can help to get relavent context from sources for user queries """
    
    docs = vector_store.similarity_search(query=query, k=5)
    
    context = ""
    for doc in docs:
        context += doc.page_content + "\n\n"
        
    return context
    
    
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
)


system_prompt = """ 
    You are a helpful assistant that answers user retrived context.
    use 'get_context' tool for questions requiring external knowledge.
""" 

In [ ]:
agent = create_agent(
    model=llm,
    tools=[get_context],
    system_prompt=system_prompt,
)

In [8]:
query = "What is the candidate name?"


res = agent.invoke({
    "messages": [
        {
            "role":"user",
            "content": query
        }
    ]
})


ai_msg = res["messages"][-1].content

ai_msg

'The candidate name is Sandip Sarkar.'